# SARBLOH Experimentation: one experiment on 5 public games (never submitted)

Daily experiment launcher. The agent is `Sarbloh-Experimentation/harness/` (a copy of `Sarbloh/harness/`; upstream
kernel `rlm/` unchanged), so experiments never touch the submission code. Each run plays the same **5 public dev
games** (2 keyboard, 2 click, 1 keyboard+click); the held-out games are never run here. Results go to
`experiments/E###_*/` and one verdict line to `experiments_<date>.md`.

Attach: the competition (ARC wheels + games + the RTX Pro 6000), `banwait13/sarblohagent` (this repo, updated after
the push), `banwait13/qwencoder` (nvidia/Qwen3.8-27B-NVFP4), `banwait13/sglangwheels` (SGLang wheels), and for the
vLLM FP8 fallback `fumiyauchiyama/qwen3-8-27b-fp8-hf-snapshot` and `banwait13/sarbloh-vllm-wheelhouse`.
Internet **off**.

## 0. CONFIG: the experiment id and its config diff (overrides of `harness/config.py`)

In [ ]:
import time
NOTEBOOK_START = time.time()

EXPERIMENT = "E118_level_skills"   # set per run: the experiments/E###_* directory this run belongs to
# the experiment's config diff, e.g. {"agent": {"perception": {"image": False}}}
# 3 games at once (keyboard ls20 tr87, click lp85), no caps. Kaggle's 12 h limit is the only clock (11.5 h budget); the
# move, turn and token limits are set far above anything a run can reach. Server: BASE's chain (SGLang NVFP4 first).
EXPERIMENT_OVERRIDES = {
    "games": ["ls20", "tr87", "lp85"], "concurrency": 3, "notebook_budget_s": 41400.0, "game_wall_s": 41400.0,
    "max_actions_per_game": 100000,
    "agent": {"limits": {"max_turns": 1000000, "max_output_tokens": 10000000000}},
}
# E111 + E110 (experiments/E111_priority_scheduler/config.yaml), the next run:
# EXPERIMENT = "E111_priority_scheduler"
# EXPERIMENT_OVERRIDES = {
#     "games": ["bp35", "cn04", "dc22", "ka59", "lf52", "lp85", "ls20", "r11l", "re86", "sb26", "sc25", "sp80",
#               "su15", "tn36", "tr87", "tu93", "wa30"],
#     "concurrency": 17, "notebook_budget_s": 18000.0, "game_wall_s": 18000.0,
#     "scheduler": {"enabled": True, "slots": 6, "quantum_calls": 4, "token_scale": 80000.0},
#     "agent": {"prompt_version": "e110"},
# }

# The 5 public dev games, fixed for every experiment so runs compare (eval/splits/heldout_split.json strata):
# keyboard ls20 tr87, click lp85 su15, keyboard+click tu93. ls20/tr87/lp85/tu93 are the standing regression gates.
GAMES = ["ls20", "tr87", "lp85", "su15", "tu93"]
HELDOUT = {"ar25", "cd82", "ft09", "g50t", "m0r0", "s5i5", "sk48", "vc33"}
assert not HELDOUT & set(GAMES), "held-out games are never run here"

BASE = {
    "model": "qwen",                  # harness/llm/qwen.py
    "experiment": EXPERIMENT,
    "games": GAMES,
    "concurrency": len(GAMES),        # all 5 at once
    "notebook_budget_s": 7200.0,      # 2 h incl. vLLM start (UNCONFIRMED: enough for 5 games to show a signal)
    "teardown_reserve_s": 300.0,
    "game_wall_s": 5400.0,
    "max_actions_per_game": 400,
    "record": True,                   # ARC SDK recording + agent step refs -> prime_run/trace.md
    # NVFP4 + MTP on SGLang first; each profile that fails start or a smoke test hands over to the next
    "server": {"profile_chain": ["sglang_nvfp4_mtp5_vision", "sglang_nvfp4_mtp_vision", "sglang_nvfp4_vision",
                               "sglang_nvfp4", "qwen_fp8_vision", "qwen_fp8"]},
}

def merge(a, b):
    out = dict(a)
    for k, v in b.items():
        out[k] = merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out

OVERRIDES = merge(BASE, EXPERIMENT_OVERRIDES)
AGENT_DATASET = "banwait13/sarblohagent"
AGENT_DIR = "Sarbloh-Experimentation"

## 1. ARC runtime (competition wheels, offline) and the agent source

In [ ]:
import os, subprocess, sys
from pathlib import Path

COMPETITION_DIR = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-warn-conflicts",
                       "--disable-pip-version-check", "--find-links", str(COMPETITION_DIR / "arc_agi_3_wheels"), "arc-agi"],
                      stdout=subprocess.DEVNULL)
# arc-agi brings its own Pillow, installed over the one Kaggle already imported (matplotlib at kernel start). Drop
# the stale modules so PIL reloads from disk as one version (else: "cannot import name '_Ink' from 'PIL._typing'").
for _m in [m for m in sys.modules if m == "PIL" or m.startswith("PIL.")]:
    del sys.modules[_m]

owner, slug = AGENT_DATASET.split("/")
candidates = [Path("/kaggle/input/datasets") / owner / slug, Path("/kaggle/input") / slug]
AGENT_ROOT = next((p for p in candidates if (p / AGENT_DIR / "harness").exists()), None)
if AGENT_ROOT is None:
    hits = [p.parent.parent for p in Path("/kaggle/input").glob(f"**/{AGENT_DIR}/harness/run.py")]
    AGENT_ROOT = hits[0].parent if hits else None
if AGENT_ROOT is None:
    raise FileNotFoundError(f"{AGENT_DIR} not found: attach {AGENT_DATASET} (updated from the git repo)")
for p in (AGENT_ROOT / AGENT_DIR, AGENT_ROOT):
    sys.path.insert(0, str(p))
print("agent root:", AGENT_ROOT)
print(subprocess.run(["ls", "-la", str(AGENT_ROOT / AGENT_DIR)], capture_output=True, text=True).stdout)

## 2. Run: vLLM up (profile chain + smoke test), play the games, teardown

In [ ]:
from harness.run import main

SUMMARY = main(OVERRIDES, notebook_start=NOTEBOOK_START)

## 3. Summary

In [ ]:
import json
print(json.dumps(SUMMARY, indent=1, default=str))
print(f"notebook wall: {time.time() - NOTEBOOK_START:.0f}s")

## 4. Step trace: per game `steps.md` (turn by turn), `steps.jsonl`, SDK recording, SDK scorecard

In [ ]:
trace_md = next(Path("/kaggle/working").rglob("prime_run/trace.md"), None)
if trace_md is None:
    print("no trace.md (run crashed before results.json?)")
else:
    print(trace_md.read_text(encoding="utf-8"))
    run_dir = trace_md.parent
    for f in sorted(run_dir.rglob("*")):
        if f.is_file() and f.suffix in (".md", ".jsonl", ".json"):
            print(f"{f.stat().st_size:>10}  {f.relative_to(run_dir)}")